# 4. Designing an intervention

Which constant dose puts every cell of a tissue into the polarized state at steady
state? The gradient comes from the implicit function theorem, so its cost does not
depend on how long the tissue takes to relax.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tlcdv/bioelectrics/blob/main/notebooks/04-designing-an-intervention.ipynb)

In [ ]:
# On Colab this installs the library. Locally it does nothing if it is already there.
try:
    import bioelectrics  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/tlcdv/bioelectrics.git

In [ ]:
import torch
from bioelectrics.ionic import IonicTissue, ChannelExpression, K_IR_CAM, CAT_DEP_CAM
from bioelectrics.interventions import ChannelBlocker, ChronicInterventionOptimizer

tissue = IonicTissue(channels=[ChannelExpression(K_IR_CAM, 1e-16),
                               ChannelExpression(CAT_DEP_CAM, 1e-16)])
v0 = torch.full((1, 6, 6), -0.078, dtype=torch.float64)
v0[:, 2:4, 2:4] = -0.003
state = tissue.with_vmem(tissue.initial_state((6, 6)), v0)

In [ ]:
opt = ChronicInterventionOptimizer(
    tissue, state, torch.full_like(v0, -0.0773),
    [ChannelBlocker('Cat_dep(CAM)', cost=1.0)])
res = opt.optimize(n_iters=40, tol_mV=1.0, verbose=True)
print()
print(res.protocol.to_markdown())

## Drugs whose potency is published

`bioelectrics.pharmacology` holds values read from primary sources, each with its
assay, species and citation.

In [ ]:
from bioelectrics import pharmacology as ph

for d in ph.LIBRARY:
    print('{:26s} {:10s} {}'.format(d.name, d.target, d.reported))

print()
for line in ph.get('ouabain').caveats():
    print('-', line)

A screen of these against a depolarized domain finds that **none of them normalizes**
it. Every compound with a verified value acts on a polarizing element, so blocking it
depolarizes. See `examples/drug_repurposing.py` and the
[limitations](https://tlcdv.github.io/bioelectrics/limitations/).